# Phase 1: Data Preparation & Validation

This phase focuses on converting unstructured election documents into structured datasets. Because real-world election data is often messy, we implemented a rigorous cleaning and validation workflow.

### Key Accomplishments:
* **Data Extraction**: Processed raw **ECT Form 5/18** (Constituency) and **Form 5/18 บช** (Party-list) into tabular format.
* **Mathematical Validation**: Verified data integrity using the official ballot balance formula:
$$\text{ballots\_used} = \text{ballots\_good} + \text{ballots\_spoiled} + \text{ballots\_no\_vote}$$
* **Missing Value Recovery**: 
    * Recovered `ballots_good` by cross-referencing individual candidate/party vote sums.
    * Imputed `eligible_voters` using a **71% national turnout baseline** to ensure a complete dataset for analysis.
* **Standardization**: Unified the party-list schema by merging `party_name` into the `entity_name` column.
* **Pipeline Organization**: Structured the project to save validated outputs to `data/clean_data/` (external to the notebooks directory) for seamless integration with the **Streamlit** dashboard.

### Final Datasets:
1. `5_18_station.csv`: Polling station metadata and ballot statistics of 5/18.
2. `5_18_party_station.csv`: Polling station metadata and ballot statistics of 5/18 บช.
3. `5_18_votes.csv`: Candidate-level constituency results.
4. `5_18_party_vote.csv`: Party-level party-list results.


In [64]:
import pandas as pd
import numpy as np

In [65]:
stations_df = pd.read_csv("../data/processed/stations.csv")
votes_df = pd.read_csv("../data/processed/votes.csv")

In [66]:
col_to_drop = ['source_pdf', 'source_pages', 'ocr_status','ballots_received']
stations_df.drop(columns=col_to_drop, inplace=True)

In [67]:
stations_df.drop_duplicates(inplace=True)
stations_df.dropna(subset=['station_code'], inplace=True)

In [68]:
station = stations_df.copy()

In [69]:
stations_df.fillna({'ballots_no_vote': 0, 'ballots_spoiled': 0}, inplace=True)

In [70]:
def heal_election_data(stations_df, votes_df):
    # 1. Calculate the 'true' ballots_good from the votes breakdown
    votes_sum = votes_df.groupby('station_code')['votes'].sum().reset_index()
    votes_sum.rename(columns={'votes': 'calculated_good_ballots'}, inplace=True)
    
    # Merge calculation back to stations
    df = stations_df.merge(votes_sum, on='station_code', how='left')
    
    # 2. Fill missing ballots_good with the calculated sum
    mask_good = df['ballots_good'].isnull()
    df.loc[mask_good, 'ballots_good'] = df['calculated_good_ballots']
    df.loc[mask_good, 'validation_flags'] = df['validation_flags'].fillna('') + "|Recovered_Good_from_Votes"
    
    # Ballots Used = Good + Spoiled + No Vote
    mask_used = df['ballots_used'].isnull()
    df.loc[mask_used, 'ballots_used'] = (
        df['ballots_good'] + 
        df['ballots_spoiled'] + 
        df['ballots_no_vote']
    )
    df.loc[mask_used, 'validation_flags'] = df['validation_flags'].fillna('') + "|Calculated_Used_via_Sum"
    
    # 4. Handle voters_present (Assuming 1:1 with ballots used if missing)
    mask_present = df['voters_present'].isnull()
    df.loc[mask_present, 'voters_present'] = df['ballots_used']
    df.loc[mask_present, 'validation_flags'] = df['validation_flags'].fillna('') + "|Estimated_Present_from_Used"
    
    # 5. Clean up temporary columns
    df.drop(columns=['calculated_good_ballots'], inplace=True)
    
    return df

# Apply the healing function
stations_df = heal_election_data(stations_df, votes_df)


In [71]:
# Calculate the actual mean turnout for YOUR constituency first
known_data = stations_df.dropna(subset=['eligible_voters', 'voters_present'])
constituency_turnout_rate = (known_data['voters_present'] / known_data['eligible_voters']).mean()

# If the constituency mean is unavailable, fallback to your 0.71 research
final_rate = constituency_turnout_rate if not np.isnan(constituency_turnout_rate) else 0.71

# Apply the estimation
mask = stations_df['eligible_voters'].isnull()
stations_df.loc[mask, 'eligible_voters'] = (stations_df['voters_present'] / final_rate).round()

# Tag the flag for the BI Dashboard
stations_df.loc[mask, 'validation_flags'] = (
    stations_df['validation_flags'].fillna('') + f"|ESTIMATED_ELIGIBLE_{final_rate:.2%}_TURNOUT"
)

In [72]:
stations_df.shape
stations_df.isnull().sum()

province             0
constituency_no      0
form_type            0
station_code         0
station_no           0
district             0
subdistrict          0
eligible_voters      0
voters_present       0
ballots_used         0
ballots_good         0
ballots_spoiled      0
ballots_no_vote      0
validation_flags    49
dtype: int64

In [73]:
import os

# 1. Define the path: '..' moves up from 'notebooks' to the root, then into 'data/clean_data'
output_path = os.path.join('..', 'data', 'clean_data')

# 2. Create the directory if it doesn't exist
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"Directory created at: {os.path.abspath(output_path)}")

# 3. Filter and save the Station Metadata (5/18 Constituency)
station_file = os.path.join(output_path, '5_18_station.csv')
stations_df[stations_df['form_type'] == '5_18'].to_csv(station_file, index=False, encoding='utf-8-sig')

# 4. Filter and save the Party-list Votes (5/18 Party) [cite: 18, 22]
party_file = os.path.join(output_path, '5_18_party_station.csv')
stations_df[stations_df['form_type'] == '5_18_party'].to_csv(party_file, index=False, encoding='utf-8-sig')

print("Files saved successfully in the root 'data/clean_data' folder.")

Files saved successfully in the root 'data/clean_data' folder.


In [74]:
station = (
    station
    .sort_values('ballots_good', ascending=False, na_position='last')
    .drop_duplicates(subset=['station_code'], keep='first')
    .reset_index(drop=True)
)


In [75]:
votes_df.drop_duplicates(inplace=True)
votes_df.dropna(subset=['station_code'], inplace=True)
votes_df.drop(columns=['votes_thai_word', 'needs_review'], inplace=True)

In [76]:
# Fix 1: Nullify physically impossible votes (single party votes > station's ballots_good)
ballots_good_map = station.set_index('station_code')['ballots_good']
votes_df['_ballots_good'] = votes_df['station_code'].map(ballots_good_map)
impossible = (
    votes_df['votes'].notna() &
    votes_df['_ballots_good'].notna() &
    (votes_df['votes'] > votes_df['_ballots_good'])
)
print(f"Cleared {impossible.sum()} impossible vote values (votes > ballots_good)")
votes_df.loc[impossible, 'votes'] = np.nan
votes_df.drop(columns=['_ballots_good'], inplace=True)

# Fix 2: Deduplicate votes from same station appearing in multiple PDFs
# Keep the row with the highest votes for each (station_code, entity_no, form_type)
before = len(votes_df)
votes_df = (
    votes_df
    .sort_values('votes', ascending=False, na_position='last')
    .drop_duplicates(subset=['station_code', 'entity_no', 'form_type'], keep='first')
    .reset_index(drop=True)
)
print(f"Removed {before - len(votes_df)} duplicate vote rows → {len(votes_df)} remaining")

Cleared 50 impossible vote values (votes > ballots_good)
Removed 1246 duplicate vote rows → 17941 remaining


In [77]:
votes_df.shape
votes_df.isnull().sum()

province               0
constituency_no        0
form_type              0
station_code           0
station_no             0
district               0
subdistrict            0
entity_kind            0
entity_no              0
entity_name            0
party_name         15939
votes               3548
dtype: int64

In [78]:
# 5_18_votes will contain constituency/candidate data
votes_5_18 = votes_df[votes_df['form_type'] == '5_18'].copy()

# 5_18_party_vote will contain party-list data
party_5_18 = votes_df[votes_df['form_type'] == '5_18_party'].copy()

party_5_18.drop(columns=['party_name'], inplace=True)

votes_5_18.to_csv(os.path.join(output_path, '5_18_votes.csv'), index=False, encoding='utf-8-sig')
party_5_18.to_csv(os.path.join(output_path, '5_18_party_vote.csv'), index=False, encoding='utf-8-sig')

print(f"Files saved to: {os.path.abspath(output_path)}")
print(f"5_18_votes rows: {len(votes_5_18)}")
print(f"5_18_party_vote rows: {len(party_5_18)}")

Files saved to: d:\DSDE\ProjectDSDE_election2026\data\clean_data
5_18_votes rows: 2002
5_18_party_vote rows: 15939


In [79]:
import pandas as pd
from sklearn.impute import KNNImputer

In [84]:
# ==========================================
# IMPUTE PARTY LIST VOTES
# ==========================================

# 1. Load data
party_vote_df = pd.read_csv("../data/clean_data/5_18_party_vote.csv")
stations_df = pd.read_csv("../data/clean_data/5_18_station.csv")

# 2. Merge station stats (fix: drop duplicate stations before merging!)
unique_stations = stations_df[['station_code', 'eligible_voters', 'ballots_good']].drop_duplicates(subset=['station_code'])

ml_party_df = party_vote_df.merge(
    unique_stations, 
    on='station_code', 
    how='left'
)

# 3. Define features for the AI
features = ['entity_no', 'eligible_voters', 'ballots_good', 'votes']
train_data = ml_party_df[features]

print(f"Party votes missing BEFORE ML: {ml_party_df['votes'].isna().sum()}")

# 4. Run KNN Imputer
imputer = KNNImputer(n_neighbors=10, weights='uniform')
imputed_data = imputer.fit_transform(train_data)

# 5. Extract predicted votes, round them, and fill the missing (NaN) values
predicted_votes = imputed_data[:, 3].round()
ml_party_df['votes'] = ml_party_df['votes'].fillna(pd.Series(predicted_votes))

print(f"Party votes missing AFTER ML: {ml_party_df['votes'].isna().sum()}")

# 6. Clean up and save the final dataset
ml_party_df = ml_party_df.drop(columns=['eligible_voters', 'ballots_good'])
ml_party_df.to_csv("../data/clean_data/5_18_party_vote_KNNImputed.csv", index=False)
print("Saved KNN-imputed party votes to: ../data/clean_data/5_18_party_vote_KNNImputed.csv\n")

Party votes missing BEFORE ML: 3423
Party votes missing AFTER ML: 0
Saved KNN-imputed party votes to: ../data/clean_data/5_18_party_vote_KNNImputed.csv



In [85]:
# ==========================================
# IMPUTE CANDIDATE / CONSTITUENCY VOTES
# ==========================================

# 1. Load data
candidate_vote_df = pd.read_csv("../data/clean_data/5_18_votes.csv")
# (No need to reload stations_df, we can reuse unique_stations from above)

# 2. Merge station stats (using the deduplicated stations)
ml_candidate_df = candidate_vote_df.merge(
    unique_stations, 
    on='station_code', 
    how='left'
)

# 3. Define features for the AI
features = ['entity_no', 'eligible_voters', 'ballots_good', 'votes']
train_data = ml_candidate_df[features]

print(f"Candidate votes missing BEFORE ML: {ml_candidate_df['votes'].isna().sum()}")

# 4. Run KNN Imputer
imputer = KNNImputer(n_neighbors=10, weights='uniform')
imputed_data = imputer.fit_transform(train_data)

# 5. Extract predicted votes, round them, and fill the missing (NaN) values
predicted_votes = imputed_data[:, 3].round()
ml_candidate_df['votes'] = ml_candidate_df['votes'].fillna(pd.Series(predicted_votes))

print(f"Candidate votes missing AFTER ML: {ml_candidate_df['votes'].isna().sum()}")

# 6. Clean up and save the final dataset
ml_candidate_df = ml_candidate_df.drop(columns=['eligible_voters', 'ballots_good'])
ml_candidate_df.to_csv("../data/clean_data/5_18_votes_KNNImputed.csv", index=False)
print("Saved KNN-imputed candidate votes to: ../data/clean_data/5_18_votes_KNNImputed.csv")

Candidate votes missing BEFORE ML: 125
Candidate votes missing AFTER ML: 0
Saved KNN-imputed candidate votes to: ../data/clean_data/5_18_votes_KNNImputed.csv
